In [ ]:
%pip install pandas sqlalchemy ipython-sql jupysql matplotlib
%pip install "prettytable>=3.12.0"

# Basic Demo: One-Day Sari-Sari Store Sales Calculator

This notebook demonstrates the Basic goal of the Sari-Sari Store Simulator.

The Basic goal is to:

- Load one day of transactions
- Load inventory data
- Calculate revenue
- Calculate expenses
- Calculate gross profit
- Track remaining stock
- Create a ledger summary
- Save results to CSV
- Save results to SQLite
- Inspect results using SQL Magic

## 1. Set up project paths

Because this notebook is inside the `notebooks/` folder, we need to point Python back to the project root so it can import modules from `src/`.

In [ ]:
from pathlib import Path
import sys

# The notebook is inside LT6_Final_Project/notebooks/
# Therefore, the project root is one folder above the current notebook folder.
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path so imports like src.basic.sales_calculator work.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
# Define important project paths

INVENTORY_PATH = PROJECT_ROOT / "data/raw/inventory.csv"
TRANSACTIONS_PATH = PROJECT_ROOT / "data/raw/transactions.csv"

OUTPUT_FOLDER = PROJECT_ROOT / "data/processed/basic"
DATABASE_PATH = PROJECT_ROOT / "src/database/sari_sari_store.db"

print("Inventory path:", INVENTORY_PATH)
print("Transactions path:", TRANSACTIONS_PATH)
print("Output folder:", OUTPUT_FOLDER)
print("Database path:", DATABASE_PATH)

In [ ]:
# Check that required input files exist

print("inventory.csv exists:", INVENTORY_PATH.exists())
print("transactions.csv exists:", TRANSACTIONS_PATH.exists())

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f"Missing file: {INVENTORY_PATH}")

if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(f"Missing file: {TRANSACTIONS_PATH}")

## 2. Preview raw input CSV files

The Basic version uses two CSV files:

1. `inventory.csv` — product-level stock, cost, and price data
2. `transactions.csv` — one day of sales transactions

In [ ]:
import pandas as pd

inventory_raw = pd.read_csv(INVENTORY_PATH)
transactions_raw = pd.read_csv(TRANSACTIONS_PATH)

display(inventory_raw.head())
display(transactions_raw.head())

In [ ]:
print("Inventory shape:", inventory_raw.shape)
print("Transactions shape:", transactions_raw.shape)

print("\nInventory columns:")
print(list(inventory_raw.columns))

print("\nTransactions columns:")
print(list(transactions_raw.columns))

## 3. Load and validate the CSV files

Now we use the Basic data loader functions from `src/basic/data_loader.py`.

These functions validate:

- Required columns
- Missing values
- Invalid dates
- Invalid numeric values
- Duplicate product IDs
- Duplicate transaction IDs
- Product IDs in transactions that are missing from inventory

In [ ]:
from src.basic.data_loader import (
    load_inventory,
    load_transactions,
    validate_transactions_match_inventory,
)

inventory = load_inventory(INVENTORY_PATH)
transactions = load_transactions(TRANSACTIONS_PATH)

validate_transactions_match_inventory(
    transactions=transactions,
    inventory=inventory,
)

print("Data loaded and validated successfully.")

In [ ]:
display(inventory.head())
display(transactions.head())

In [ ]:
inventory.info()

In [ ]:
transactions.info()

## 4. Run the Basic sales calculations step by step

The next cells show each major part of the calculation:

1. Join transactions with inventory
2. Calculate revenue, expenses, and gross profit
3. Summarize results by product
4. Create a daily ledger summary

In [ ]:
from src.basic.sales_calculator import (
    create_sales_details,
    create_product_summary,
    create_ledger_summary,
    print_basic_report,
)

sales_details = create_sales_details(
    inventory=inventory,
    transactions=transactions,
)

display(sales_details.head())

In [ ]:
# Show selected transaction-level columns

display(
    sales_details[
        [
            "transaction_id",
            "transaction_date",
            "product_id",
            "product_name",
            "category",
            "quantity_sold",
            "unit_price",
            "unit_cost",
            "revenue",
            "expense",
            "gross_profit",
        ]
    ].head(10)
)

In [ ]:
product_summary = create_product_summary(sales_details)

display(product_summary)

In [ ]:
ledger_summary = create_ledger_summary(
    sales_details=sales_details,
    product_summary=product_summary,
)

display(ledger_summary)

In [ ]:
print_basic_report(
    ledger_summary=ledger_summary,
    product_summary=product_summary,
)

## 5. Run the full Basic workflow

The full workflow function performs all the same steps automatically and saves outputs to:

- `data/processed/basic/`
- `src/database/sari_sari_store.db`

In [ ]:
from src.basic.sales_calculator import run_basic_sales_calculator

sales_details, product_summary, ledger_summary = run_basic_sales_calculator(
    inventory_csv_path=INVENTORY_PATH,
    transactions_csv_path=TRANSACTIONS_PATH,
    output_folder=OUTPUT_FOLDER,
    sqlite_db_path=DATABASE_PATH,
    save_csv_outputs=True,
    save_sqlite_database=True,
)

## 6. Check generated CSV output files

In [ ]:
generated_files = [
    OUTPUT_FOLDER / "daily_transaction_details.csv",
    OUTPUT_FOLDER / "daily_product_summary.csv",
    OUTPUT_FOLDER / "daily_ledger_summary.csv",
]

for file_path in generated_files:
    print(file_path, "exists:", file_path.exists())

In [ ]:
daily_transaction_details_csv = pd.read_csv(
    OUTPUT_FOLDER / "daily_transaction_details.csv"
)

daily_product_summary_csv = pd.read_csv(
    OUTPUT_FOLDER / "daily_product_summary.csv"
)

daily_ledger_summary_csv = pd.read_csv(
    OUTPUT_FOLDER / "daily_ledger_summary.csv"
)

display(daily_transaction_details_csv.head())
display(daily_product_summary_csv)
display(daily_ledger_summary_csv)

## 7. Connect to SQLite using SQL Magic

The Python script saved the tables into this SQLite database:

`src/database/sari_sari_store.db`

Now we inspect the tables with SQL Magic.

In [ ]:
%load_ext sql

In [ ]:
# Connect SQL Magic to the SQLite database.
# Because this notebook is inside notebooks/, use ../ to go back to the project root.

%sql sqlite:///../src/database/sari_sari_store.db

In [ ]:
%%sql

SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;

## 8. Inspect SQLite tables

In [ ]:
%%sql

SELECT *
FROM inventory
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM transactions
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM daily_transaction_details
LIMIT 10;

In [ ]:
%%sql

SELECT *
FROM daily_product_summary
ORDER BY product_id;

In [ ]:
%%sql

SELECT *
FROM daily_ledger_summary;

## 9. SQL ledger check

This query recalculates total revenue, total expense, and gross profit directly from the base tables:

- `transactions`
- `inventory`

This verifies that the stored ledger matches the raw data.

In [ ]:
%%sql

SELECT
    SUM(t.quantity_sold * i.unit_price) AS total_revenue,
    SUM(t.quantity_sold * i.unit_cost) AS total_expense,
    SUM(t.quantity_sold * i.unit_price) - SUM(t.quantity_sold * i.unit_cost) AS gross_profit
FROM transactions AS t
JOIN inventory AS i
    ON t.product_id = i.product_id;

## 10. SQL stock check

This query checks whether each product still has enough stock after the day's sales.

In [ ]:
%%sql

SELECT
    i.product_id,
    i.product_name,
    i.category,
    i.starting_stock,
    COALESCE(SUM(t.quantity_sold), 0) AS total_quantity_sold,
    i.starting_stock - COALESCE(SUM(t.quantity_sold), 0) AS remaining_stock,
    CASE
        WHEN i.starting_stock - COALESCE(SUM(t.quantity_sold), 0) >= 0
            THEN 'OK'
        ELSE 'INSUFFICIENT STOCK'
    END AS stock_status
FROM inventory AS i
LEFT JOIN transactions AS t
    ON i.product_id = t.product_id
GROUP BY
    i.product_id,
    i.product_name,
    i.category,
    i.starting_stock
ORDER BY i.product_id;

## 11. Product performance checks

In [ ]:
%%sql

SELECT
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue,
    total_expense,
    total_gross_profit,
    remaining_stock,
    stock_status
FROM daily_product_summary
ORDER BY total_revenue DESC;

In [ ]:
%%sql

SELECT
    category,
    SUM(total_quantity_sold) AS category_quantity_sold,
    SUM(total_revenue) AS category_revenue,
    SUM(total_expense) AS category_expense,
    SUM(total_gross_profit) AS category_gross_profit
FROM daily_product_summary
GROUP BY category
ORDER BY category_revenue DESC;

## 12. Simple visualizations

The next cells create basic charts from the product summary.

In [ ]:
import matplotlib.pyplot as plt

product_summary_sorted = product_summary.sort_values(
    by="total_revenue",
    ascending=False,
)

plt.figure(figsize=(10, 5))
plt.bar(
    product_summary_sorted["product_name"],
    product_summary_sorted["total_revenue"],
)

plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
stock_summary_sorted = product_summary.sort_values(
    by="remaining_stock",
    ascending=True,
)

plt.figure(figsize=(10, 5))
plt.bar(
    stock_summary_sorted["product_name"],
    stock_summary_sorted["remaining_stock"],
)

plt.title("Remaining Stock by Product")
plt.xlabel("Product")
plt.ylabel("Remaining Stock")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
category_summary = (
    product_summary
    .groupby("category", as_index=False)
    .agg(
        category_revenue=("total_revenue", "sum"),
        category_expense=("total_expense", "sum"),
        category_gross_profit=("total_gross_profit", "sum"),
    )
)

display(category_summary)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    category_summary["category"],
    category_summary["category_gross_profit"],
)

plt.title("Gross Profit by Category")
plt.xlabel("Category")
plt.ylabel("Gross Profit")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 13. Final Basic goal validation

The Basic goal is successful if:

- Revenue is calculated
- Expenses are calculated
- Gross profit is calculated
- Remaining stock is calculated
- No unexpected stock issues exist
- The ledger status is `BALANCED`
- Output CSV files are created
- SQLite tables are created

In [ ]:
ledger_status = ledger_summary.loc[0, "ledger_status"]

print("Ledger status:", ledger_status)

if ledger_status == "BALANCED":
    print("Basic goal completed successfully.")
else:
    print("Basic goal needs review.")

In [ ]:
# Final checklist

checks = {
    "inventory_loaded": not inventory.empty,
    "transactions_loaded": not transactions.empty,
    "sales_details_created": not sales_details.empty,
    "product_summary_created": not product_summary.empty,
    "ledger_summary_created": not ledger_summary.empty,
    "database_created": DATABASE_PATH.exists(),
    "transaction_details_csv_created": (OUTPUT_FOLDER / "daily_transaction_details.csv").exists(),
    "product_summary_csv_created": (OUTPUT_FOLDER / "daily_product_summary.csv").exists(),
    "ledger_summary_csv_created": (OUTPUT_FOLDER / "daily_ledger_summary.csv").exists(),
}

checks

## End of Basic Demo

This notebook demonstrated the Basic level of the Sari-Sari Store Simulator.

The main project logic is stored in `.py` files under `src/basic/`.

This notebook is only used for demonstration, validation, SQL Magic inspection, and simple visualizations.